# 01 Smoothing Context and Model Choice

Smoothing methods reduce rapid fluctuations in a time series so that the underlying level, trend, and seasonality are easier to forecast. The key modeling idea is that recent observations usually deserve more weight than older observations, but the amount of weight should depend on how quickly the process changes.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check_between, check_close, check_columns
from smoothing_utils import (
    accuracy_measures, initial_level_mean, initial_line,
    simple_es, optimize_simple_es,
    holt_trend, optimize_holt, holt_forecast_table,
    holt_winters, optimize_holt_winters, holt_winters_forecast,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')

## The Three Families in This Module

| Data pattern | Recommended model | State components |
|---|---|---|
| No clear trend and no seasonal pattern, but the level may drift | Simple exponential smoothing | Level only |
| Linear trend but no seasonal pattern | Holt's trend-corrected exponential smoothing | Level and trend |
| Linear trend with seasonal pattern | Holt-Winters exponential smoothing | Level, trend, and seasonal factors |

Additive Holt-Winters is used when seasonal swings are roughly constant. Multiplicative Holt-Winters is used when seasonal swings grow as the series level grows.

## Software Model Names

Forecasting software often labels these same ideas with ETS notation, where the three letters describe the error, trend, and seasonal components.

| Course name | Common ETS label | Meaning |
|---|---|---|
| Simple exponential smoothing | ETS(A,N,N), often written as ANN | additive errors, no trend, no seasonality |
| Holt's trend-corrected method | ETS(A,A,N), often written as AAN | additive errors, additive trend, no seasonality |
| Additive Holt-Winters | ETS(A,A,A), often written as AAA | additive errors, additive trend, additive seasonality |
| Multiplicative seasonal Holt-Winters | often written with multiplicative seasonality, such as MAM | seasonal effects scale with the level |

Different software packages may use slightly different initialization and numerical optimization details. Small differences in constants are normal; the important checks are that the model family, equations, and interpretation match the data pattern.


In [ ]:
series = {
    'Cod catch: level only': pd.read_csv(DATA_DIR / 'cod_catch.csv').set_index('Time')['CodCatch'],
    'Thermostat sales: trend': pd.read_csv(DATA_DIR / 'thermostat_sales.csv').set_index('Week')['ThermostatSales'],
    'Bike sales: additive seasonality': pd.read_csv(DATA_DIR / 'bike_sales.csv').set_index('Time')['BikeSales'],
    'Sports drink sales: multiplicative seasonality': pd.read_csv(DATA_DIR / 'sports_drink_sales.csv').set_index('Time')['DrinkSales'],
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (title, values) in zip(axes.ravel(), series.items()):
    ax.plot(values.index, values.values, marker='o')
    ax.set_title(title)
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
plt.tight_layout()

## What the Plots Tell Us

- Cod catch fluctuates around a slowly changing level, so a level-only model is a reasonable starting point.
- Thermostat sales has a directional pattern without obvious seasonality, so Holt's trend-corrected model is designed for this setting.
- Bike sales repeats quarterly swings of similar size, suggesting additive seasonal factors.
- Sports drink sales has seasonal swings that become larger as the level rises, suggesting multiplicative seasonal factors.

In [ ]:
pd.DataFrame({
    'series': list(series.keys()),
    'n_observations': [len(v) for v in series.values()],
    'min': [v.min() for v in series.values()],
    'max': [v.max() for v in series.values()],
})

## Practical Rule

Choose the simplest model that captures the visible structure. A more complex model can fit noise if it is used only because the software makes it easy to click. We will still compare accuracy measures, but model choice begins with the time-series plot.